### Import packages
#### Create connection to Database

In [2]:
from sqlalchemy import create_engine
import pandas as pd

# Connect to the database
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()

#### Read table from database

In [ ]:
# HINTS
# Load data into DataFrame
# user Pandas to read data from a table into a DataFrame

In [ ]:
# Approach 1: Use Pandas.read_sql_table to read all columns from 'customers' table
# table_name = 'customers'
# df = pd.read_sql_table(table_name, con=db_conn)
# df.tail(5)

In [ ]:
# # Approach 2: Use Pandas.read_sql_query to read these columns
# table_name = 'customers'
# columns = ['CustomerId', 'FirstName', 'LastName', 'Phone', 'Email', 'SupportRepId']
# table_name = 'customers'
# df = pd.read_sql_query(sql='select * from customers',con=db_conn)
# df.tail
# df.to_csv('customers.csv')


#### Config-Driven Ingestion

In [12]:
# # HINTS
# # Read configs stored in the 'config.yml' file

# # Read yaml file
# # Package: yaml (pip install pyyaml)
# # Function: load / safe_load
# # Print it after loading

# import json

import os
import io
import yaml

config_file = 'config.yml'
f = open(config_file, 'r')
config = yaml.safe_load(f)
config

{'source': {'database': {'host': 'chinook', 'db_type': 'sqlite'},
  'table': ['albums',
   'artists',
   'customers',
   'employees',
   'genres',
   'invoice_items',
   'invoices',
   'media_types',
   'playlist_track',
   'playlists',
   'tracks']}}

In [4]:
# Use loop function to read tables within config.source.table
# Export output into CSV
# Name Convention: '<date>__<table_name>.csv'
# Path: chinook/config_driven/
# note: use os.makedirs() if path is not exists

def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv')
    print('Completed!\n')

def get_connection(db_type, host):
    if db_type == 'sqlite':
        db_connection_string = f'sqlite:///{host}.db'
        db_engine = create_engine(url=db_connection_string)
        return db_engine.connect()
    elif db_type == 'Oracle':
        db_connection_string = 'Oracle://{host}:1234'
        return db_engine.connect()
db_conn = get_connection(**config.get('source').get('database'))
extract_table(table_name='albums', con=db_conn, folder_path='destination/config_driven')

Extracting albums ...
Completed!



### Metadata-Driven Ingestion

In [16]:
# # HINTS
# # Read metadata from the database inlcuding tables / columns
sqlite_metadata_table = 'sqlite_master'
sqlite_metadata_condition = "type = 'table'"
# metadata_sql = f""" select 1"""
metadata_sql="select tbl_name from " + sqlite_metadata_table + " where " + sqlite_metadata_condition

import os
table_df = pd.read_sql_query(metadata_sql,con=db_conn)
folder_path='destination/metadata_driven'
os.makedirs(folder_path)
for i in range(len(table_df)):
    table_name = str(table_df.loc[i].tbl_name)
    print(table_name)
    # df = pd.read_sql_table(table_name, con=db_conn)
    df = pd.read_sql_query(str("select * from " + table_name), con=db_conn)
    export_file_name=table_name+'.csv'  
    df.to_csv(f'{folder_path}/{export_file_name}', index=False)
    # df.to_csv(export_file_name,index=False)
    # extract_table(table_name='albums', con=db_conn, folder_path='destination/config_driven')
    df.drop(df.columns, axis=1, inplace=True)
    


albums
sqlite_sequence
artists
customers
employees
genres
invoices
invoice_items
media_types
playlists
playlist_track
tracks
sqlite_stat1


In [ ]:
# loop for each table from the DataFrame
# read and extract table
# save to path: destination/metadata_driven/
# note: use os.makedirs() if path is not exists

In [ ]:


extract_table(table_name=table_name, con=db_conn, folder_path='destination/config_driven')
# import os

# import yaml
# import io
config_file = 'config.yml'
f = open(config_file, 'r')
config = yaml.safe_load(f)
config

def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=con)
    df.to_csv(f'{folder_path}/20251127_{table_name}.csv',index=False)
    print('Completed!\n')

def get_connection(db_type, host):
    if db_type == 'sqlite':
        db_connection_string = f'sqlite:///{host}.db'
        db_engine = create_engine(url=db_connection_string)
        return db_engine.connect()
    elif db_type == 'Oracle':
        db_connection_string = 'Oracle://{host}:1234'
        return db_engine.connect()
db_conn = get_connection(**config.get('source').get('database'))

for table_name in config.get('source').get('table'):
    extract_table(table_name=table_name, con=db_conn, folder_path='destination/config_driven')

In [ ]:
metadata_sql = """select name from sqlite_master where 1=1 and type = 'table'"""
table_df = pd.read_sql_query(metadata_sql, con=db_conn)
names = [tb for tb in list(table_df['name']) if tb not in ('sqlite_stat1', 'sqlite_sequence')]
import os 


# loop for each table from the DataFrame
# read and extract table
# save to path: chinook/metadata_driven/
# note: use os.makedirs() if path is not exists
def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv', index=False)
    print('Completed!\n')
for name in names:
    extract_table(table_name=name, con=db_conn, folder_path='destination/metadata')

In [ ]:
from sqlalchemy import create_engine, inspect
import pandas as pd
import os
# Connect to the database
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()
db_inspector = inspect(db_engine)

tables = db_inspector.get_table_names()
print("Table found: ", tables)

output_folder = "./exports"
os.makedirs(output_folder, exist_ok=True)
for table_name, df in dataFrames.items():
    file_path = os.path.join(output_folder)
    df.to_csv(f"./exports/{table_name}.csv", index = False)
    print(f"Exported: {file_path}")